In [ ]:
"""
=============================================================================
  STFT CNN for ECG Classification & Arrhythmia Detection at the Edge
=============================================================================
 
Pipeline
--------
STEP 1 -> Load MIT-BIH dataset and map beats to AAMI classes
STEP 2 -> Resample to 128 Hz and segment beats into 64-sample windows
STEP 3 -> Build FIR filter-bank (pre-initialised Conv1D kernels)
STEP 4 -> Define two model architectures (Conv1D_T and Conv1D_2D_T)
STEP 5 -> Handle class imbalance via class weights
STEP 6 -> Train with Keras (Adam + callbacks)

STEP 7 -> Quantise and prune for edge deployment (TFLite)
STEP 8 -> Benchmark on edge device (inference time, throughput, accuracy) 
"""

In [ ]:
# =============================================================================
# Imports
# =============================================================================
 
import time
import numpy as np
from math import gcd
import os
 
import wfdb                                     # MIT-BIH reader
from scipy.signal import firwin, resample_poly  # FIR design + resampling
import matplotlib.pyplot as plt
 
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
 
import tensorflow as tf

keras = tf.keras
layers = tf.keras.layers

In [ ]:
# =============================================================================
# Global constants
# =============================================================================
 
FS_ORIG   = 360    # MIT-BIH original sampling rate (Hz)
FS_TARGET = 128    # target rate after resampling (Hz)
WIN_HALF  = 32     # half-window around R-peak  =>  64 samples total
N_CLASSES = 5      # AAMI superclasses: N, S, V, F, Q
 
# Best Conv1D FIR layer parameters (found by Keras Tuner, Section V)
N_FILTERS   = 8    # number of band-pass filters (NF)
KERNEL_SIZE = 16   # FIR filter length            (NK)
N_SAMPLES   = 64   # beat segment length          (Ns)

In [ ]:
# =============================================================================
# STEP 1 -- AAMI label mapping  (Table 1 in paper)
# =============================================================================
#
#  Class 0  N  -- Normal beat
#  Class 1  S  -- Supraventricular ectopic beat
#  Class 2  V  -- Ventricular ectopic beat
#  Class 3  F  -- Fusion beat
#  Class 4  Q  -- Unknown / paced beat
 
AAMI_MAP = {
    # Normal
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
    # Supraventricular
    'A': 1, 'a': 1, 'J': 1, 'S': 1,
    # Ventricular
    'V': 2, 'E': 2,
    # Fusion
    'F': 3,
    # Unknown / paced
    '/': 4, 'f': 4, 'Q': 4,
}
 
# Records 102 and 104 are excluded (no MLII lead)
ALL_RECORDS = [
    100, 101, 103, 105, 106, 107, 108, 109,
    111, 112, 113, 114, 115, 116, 117, 118, 119,
    121, 122, 123, 124, 200, 201, 202, 203, 205,
    207, 208, 209, 210, 212, 213, 214, 215, 217,
    219, 220, 221, 222, 223, 228, 230, 231, 232,
    233, 234,
]

In [ ]:
# =============================================================================
# STEP 2 -- Load, resample and segment the MIT-BIH dataset
# =============================================================================
 
def load_mitbih(data_dir: str, records=ALL_RECORDS):
    """
    For every MIT-BIH record:
      1. Read the MLII lead signal.
      2. Resample from 360 Hz to 128 Hz using resample_poly (no aliasing).
      3. For each annotated beat, cut a 64-sample window centred at the R-peak.
      4. Extract normalised pre-RR and post-RR interval features.
 
    Parameters
    ----------
    data_dir : str
        Folder that contains the MIT-BIH .dat / .hea / .atr files.
    records : list of int
        Record numbers to load (default: all 46 valid records).
 
    Returns
    -------
    X_ecg : np.ndarray  shape (N, 64, 1)    -- beat segments
    X_rr  : np.ndarray  shape (N, 2)        -- [pre_RR, post_RR] in seconds
    y     : np.ndarray  shape (N,)          -- integer AAMI class labels
    """
    X_ecg, X_rr, y = [], [], []
 
    # -------------------------------------------------------------------------
    # Rational resampling factors:  360 Hz -> 128 Hz
    #
    # Using signal[::int(360/128)] = signal[::2] gives 180 Hz -- WRONG.
    # The correct approach is resample_poly(signal, up, down):
    #
    #   gcd(128, 360) = 8
    #   up   = 128 / 8 = 16
    #   down = 360 / 8 = 45
    #
    # resample_poly upsamples by 16, applies an anti-aliasing FIR filter,
    # then downsamples by 45.
    # Result length = round(len(signal) * 128 / 360)  -- exact 128 Hz.
    # -------------------------------------------------------------------------
    g    = gcd(FS_TARGET, FS_ORIG)  
    UP   = FS_TARGET // g            
    DOWN = FS_ORIG   // g           
 
    for rec_id in records:
        try:
            record = wfdb.rdrecord(f'{data_dir}/{rec_id}')
            annot  = wfdb.rdann(f'{data_dir}/{rec_id}', 'atr')
        except Exception as exc:
            print(f"  [skip] record {rec_id}: {exc}")
            continue
 
        # Channel 0 is MLII for all included records
        signal_orig = record.p_signal[:, 0].astype(np.float32)
 
        # Resample to 128 Hz (exact, anti-aliased)
        signal_ds = resample_poly(signal_orig, UP, DOWN).astype(np.float32)
 
        r_peaks  = annot.sample   # R-peak positions in ORIGINAL 360 Hz samples
        sym_list = annot.symbol
 
        for i, (peak_orig, sym) in enumerate(zip(r_peaks, sym_list)):
 
            if sym not in AAMI_MAP:
                continue   # skip non-beat annotations
 
            label = AAMI_MAP[sym]
 
            # Convert R-peak from 360 Hz index to 128 Hz index.
            peak_ds = int(round(peak_orig * FS_TARGET / FS_ORIG))
 
            start = peak_ds - WIN_HALF   # inclusive
            end   = peak_ds + WIN_HALF   # exclusive  ->  64 samples total
 
            # Drop beats whose window falls outside the signal boundaries
            if start < 0 or end > len(signal_ds):
                continue
 
            segment = signal_ds[start:end]   # shape (64,)
 
            # RR-interval features (in seconds, computed at 360 Hz)
            # The paper notes these significantly improve classification.
            if 0 < i < len(r_peaks) - 1:
                pre_rr  = (r_peaks[i]     - r_peaks[i - 1]) / FS_ORIG
                post_rr = (r_peaks[i + 1] - r_peaks[i])     / FS_ORIG
            else:
                pre_rr = post_rr = 0.0
 
            X_ecg.append(segment)
            X_rr.append([pre_rr, post_rr])
            y.append(label)
 
    X_ecg = np.array(X_ecg, dtype=np.float32)[:, :, np.newaxis]  # (N, 64, 1)
    X_rr  = np.array(X_rr,  dtype=np.float32)                    # (N, 2)
    y     = np.array(y,      dtype=np.int32)                     # (N,)
 
    return X_ecg, X_rr, y
 
 
def split_data(X_ecg, X_rr, y, test_size=0.25, val_size=0.25, seed=42):
    """
    Stratified random split into train / val / test sets.
 
    The paper uses a 75/25 train-test split, then a further
    75/25 train-val split, preserving class proportions in each set.
 
    Returns
    -------
    Xe_tr, Xr_tr, y_tr  -- training set
    Xe_vl, Xr_vl, y_vl  -- validation set
    Xe_te, Xr_te, y_te  -- test set
    """
    # Split 1: (train + val) vs test
    Xe_tv, Xe_te, Xr_tv, Xr_te, y_tv, y_te = train_test_split(
        X_ecg, X_rr, y,
        test_size=test_size, stratify=y, random_state=seed)
 
    # Split 2: train vs val  (from the train+val portion)
    Xe_tr, Xe_vl, Xr_tr, Xr_vl, y_tr, y_vl = train_test_split(
        Xe_tv, Xr_tv, y_tv,
        test_size=val_size, stratify=y_tv, random_state=seed)
 
    return (Xe_tr, Xr_tr, y_tr,
            Xe_vl, Xr_vl, y_vl,
            Xe_te, Xr_te, y_te)
 
 

In [ ]:
# =============================================================================
# STEP 3 -- FIR filter-bank (pre-initialised Conv1D kernel weights)
# =============================================================================
 
def build_fir_filterbank(n_filters = N_FILTERS,
                         kernel_size = KERNEL_SIZE,
                         fs = float(FS_TARGET)):
    """
    Design NF adjacent band-pass FIR filters using the Hamming window method.
 
    The filters divide [0, fs/2] into NF equal-width frequency bands.
    This mimics the STFT operation: each filter passes one frequency band
    and its output is the energy of that band over time -- exactly what
    an STFT spectrogram shows.
 
    Parameters
    ----------
    n_filters   : number of filters (NF in paper, default 8)
    kernel_size : filter tap length (NK in paper, default 16)
    fs          : sampling rate Hz  (default 128)
 
    Returns
    -------
    kernels : np.ndarray  shape (kernel_size, 1, n_filters)
              Ready to be passed to layer.set_weights([kernels]).
    """
    nyq      = fs / 2.0               # Nyquist frequency
    bw       = nyq / n_filters        # bandwidth of each band (Hz)
    kernels  = np.zeros((kernel_size, 1, n_filters), dtype=np.float32)
 
    for k in range(n_filters):
        low  = (k * bw) / nyq          # normalised lower cut-off [0, 1]
        high = ((k + 1) * bw) / nyq    # normalised upper cut-off [0, 1]
 
        # Keep cut-offs strictly inside (0, 1) to avoid firwin errors
        low  = max(low,  1e-4)
        high = min(high, 1.0 - 1e-4)
 
        # pass_zero=False -> band-pass (not low-pass)
        coeffs = firwin(kernel_size, [low, high],
                        pass_zero=False, window='hamming')
        kernels[:, 0, k] = coeffs.astype(np.float32)
 
    return kernels   # shape: (kernel_size, 1, n_filter)

In [ ]:
# =============================================================================
# STEP 4 -- Model architectures
# =============================================================================
 
def build_conv1d_model(n_filters=N_FILTERS,
                       kernel_size=KERNEL_SIZE,
                       n_samples=N_SAMPLES,
                       n_classes=N_CLASSES,
                       trainable_fir=True):
    """
    Model A: purely 1D CNN.
 
    Signal branch
    -------------
    Input(64, 1)
    -> Conv1D FIR (8 x 16, ReLU, trainable=True/False)   [STFT block]
    -> BatchNorm -> MaxPool1D(2)
    -> Conv1D(16, 9) -> BatchNorm -> ReLU -> MaxPool1D(2)
    -> Conv1D(16, 8) -> BatchNorm -> ReLU -> MaxPool1D(2)
    -> Flatten
 
    RR branch
    ---------
    Input(2,) -> Dense(32) -> Dense(16) -> Dense(8)  [all ReLU]
 
    Head
    ----
    Concatenate -> Dense(5, Softmax)
 
    trainable_fir=True  -> model name: Conv1D_T
    trainable_fir=False -> model name: Conv1D_T_FIR  (fixed FIR filters)
    """
    fir_weights = build_fir_filterbank(n_filters, kernel_size)
 
    # --- Signal branch -------------------------------------------------------
    sig_in = keras.Input(shape=(n_samples, 1), name='signal')
 
    # STFT / FIR Conv1D block (paper's core contribution)
    # When trainable_fir=True, the FIR coefficients are fine-tuned during
    # backprop, improving classification performance (Table 2 in paper).
    x = layers.Conv1D(
            filters=n_filters,
            kernel_size=kernel_size,
            padding='same',
            use_bias=False,
            trainable=trainable_fir,
            name='fir_conv1d')(sig_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Concatenate()([sig_in, x])
    x = layers.MaxPooling1D(pool_size=2)(x)
 
    x = layers.Conv1D(16, kernel_size=8, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
 
    x = layers.Conv1D(16, kernel_size=8, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
 
    sig_out = layers.Flatten()(x)
 
    # --- RR-interval branch --------------------------------------------------
    rr_in  = keras.Input(shape=(2,), name='rr_intervals')
    r      = layers.Dense(32, activation='relu')(rr_in)
    r      = layers.Dense(16, activation='relu')(r)
    r      = layers.Dense(8,  activation='relu')(r)
    rr_out = layers.Flatten()(r)
 
    # --- Merge and classify --------------------------------------------------
    merged = layers.Concatenate()([sig_out, rr_out])
    out    = layers.Dense(n_classes, activation='softmax')(merged)
 
    name  = 'Conv1D_T' if trainable_fir else 'Conv1D_T_FIR'
    model = keras.Model(inputs=[sig_in, rr_in], outputs=out, name=name)
 
    # Load pre-designed FIR coefficients as initial weights
    model.get_layer('fir_conv1d').set_weights([fir_weights])
 
    return model
 
 
def build_conv1d_2d_model(n_filters=N_FILTERS,
                          kernel_size=KERNEL_SIZE,
                          n_samples=N_SAMPLES,
                          n_classes=N_CLASSES,
                          trainable_fir=True):
    """
    Model B: Conv1D_2D_T -- best model in the paper.
 
    The FIR Conv1D front-end produces NF feature maps of length Ns.
    These are reshaped into a 2D heatmap (Ns x NF x 1) that resembles
    an STFT spectrogram, then fed into a 2D CNN classifier.
 
    Signal branch
    -------------
    Input(64, 1)
    -> Conv1D FIR (8 x 16, Tanh)    [STFT front-end, Tanh chosen by tuner]
    -> BatchNorm
    -> Reshape(64, 8, 1)             [1D feature maps -> 2D heatmap image]
    -> Conv2D(32, 4x4) -> BatchNorm -> ReLU -> MaxPool2D(2x2)
    -> Conv2D(16, 4x4) -> BatchNorm -> ReLU -> MaxPool2D(2x2)
    -> Conv2D(16, 4x4) -> BatchNorm -> ReLU
    -> Flatten
 
    RR branch  (identical to Model A)
 
    Head
    ----
    Concatenate -> Dense(5, Softmax)
 
    trainable_fir=True  -> model name: Conv1D_2D_T
    trainable_fir=False -> model name: Conv1D_2D_T_FIR
    """
    fir_weights = build_fir_filterbank(n_filters, kernel_size)
 
    # --- Signal branch -------------------------------------------------------
    sig_in = keras.Input(shape=(n_samples, 1), name='signal')
 
    # FIR / STFT Conv1D layer.
    # Tanh activation was chosen by Keras Tuner over ReLU / Sigmoid.
    x = layers.Conv1D(
            filters=n_filters,
            kernel_size=kernel_size,
            padding='same',
            use_bias=False,
            trainable=trainable_fir,
            name='fir_conv1d')(sig_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('tanh')(x)   
 
    # Reshape 1D feature maps (Ns, NF) into a 2D heatmap (Ns, NF, 1).
    # Axis 1 = time, Axis 2 = frequency band index.
    x = layers.Reshape((n_samples, n_filters, 1))(x)  # -> (64, 8, 1)
 
    # 2D CNN Block 1
    x = layers.Conv2D(32, kernel_size=(4, 4), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
 
    # 2D CNN Block 2
    x = layers.Conv2D(16, kernel_size=(4, 4), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
 
    # 2D CNN Block 3
    x = layers.Conv2D(16, kernel_size=(4, 4), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
 
    sig_out = layers.Flatten()(x)
 
    # --- RR-interval branch --------------------------------------------------
    rr_in  = keras.Input(shape=(2,), name='rr_intervals')
    r      = layers.Dense(32, activation='relu')(rr_in)
    r      = layers.Dense(16, activation='relu')(r)
    r      = layers.Dense(8,  activation='relu')(r)
    rr_out = layers.Flatten()(r)
 
    # --- Merge and classify --------------------------------------------------
    merged = layers.Concatenate()([sig_out, rr_out])
    out    = layers.Dense(n_classes, activation='softmax')(merged)
 
    name  = 'Conv1D_2D_T' if trainable_fir else 'Conv1D_2D_T_FIR'
    model = keras.Model(inputs=[sig_in, rr_in], outputs=out, name=name)
 
    # Load pre-designed FIR coefficients as initial weights
    model.get_layer('fir_conv1d').set_weights([fir_weights])
 
    return model

In [ ]:
# =============================================================================
# STEP 5 -- Class-imbalance handling
# =============================================================================
 
def get_class_weights(y_train):
    """
    Compute inverse-frequency class weights.
 
    The MIT-BIH dataset is heavily imbalanced:
    Normal (N) beats account for ~86 % of all beats.
 
    Passing class_weight to model.fit() increases the loss contribution
    of minority classes (S, V, F, Q), equivalent to oversampling them.
    The paper uses this instead of SMOTE or manual oversampling.
 
    Returns a dict  {class_index: weight}  ready for Keras.
    """
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    return dict(zip(classes.tolist(), weights.tolist()))

In [ ]:
# =============================================================================
# STEP 6 -- Training
# =============================================================================
 
def train_model(model,
                Xe_tr, Xr_tr, y_tr,
                Xe_vl, Xr_vl, y_vl,
                class_weights,
                epochs=50,
                batch_size=256):
    """
    Compile and fit the model using the paper's training setup:
      - Optimizer : Adam, initial lr = 0.01
      - Loss      : categorical cross-entropy
      - Callbacks : ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
 
    Note: the paper maximises macro F1-score as the tuning objective
    (not accuracy) because accuracy is misleading on imbalanced data --
    a classifier that always predicts 'Normal' would reach 86 % accuracy
    while completely missing arrhythmias.
 
    Parameters
    ----------
    model         : Keras model from build_conv1d_model or build_conv1d_2d_model
    Xe_tr/vl      : ECG segments  (N, 64, 1)
    Xr_tr/vl      : RR features   (N, 2)
    y_tr/vl       : integer labels (N,)
    class_weights : dict from get_class_weights()
 
    Returns
    -------
    history : Keras History object
    """
    y_tr_oh = keras.utils.to_categorical(y_tr, N_CLASSES)
    y_vl_oh = keras.utils.to_categorical(y_vl, N_CLASSES)
 
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss='categorical_crossentropy',
        metrics=['accuracy'])
 
    callbacks = [
        # Halve LR when val_loss does not improve for 5 epochs
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5,
            min_lr=1e-6, verbose=1),
 
        # Stop training when val_loss has not improved for 15 epochs
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15,
            restore_best_weights=True),
 
        # Save the best checkpoint to disk
        keras.callbacks.ModelCheckpoint(
            'best_model.keras',
            save_best_only=True,
            monitor='val_loss')]
 
    history = model.fit(
        x={'signal': Xe_tr, 'rr_intervals': Xr_tr},
        y=y_tr_oh,
        validation_data=({'signal': Xe_vl, 'rr_intervals': Xr_vl}, y_vl_oh),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1)
 
    return history

In [ ]:
# Model evaluation
def model_evaluation(model, model_name):
    
    y_pred = np.argmax(model.predict({'signal': Xe_te, 'rr_intervals': Xr_te}), axis=1)
     
    print(f"\n {model_name} Model evaluation:")
    print(classification_report(y_te, y_pred, target_names=['N', 'S', 'V', 'F', 'Q'], digits=4))
    
    # Create confusion matrix
    cm = confusion_matrix(y_te, y_pred)
    class_names = ['N', 'S', 'V', 'F', 'Q']
    
    # Plot confusion matrix
    plt.figure(figsize=(5, 3))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'Confusion Matrix - {model_name} Model')
    plt.show()


In [ ]:
# Model size analysis
def get_model_size_analysis(model):

    dtype_bytes = {'float16':2,'bfloat16':2,'float32':4,'float64':8,
                   'int8':1,'int16':2,'int32':4,'int64':8,'uint8':1,'uint16':2,'uint32':4,'uint64':8}

    W = 72
    print(f"\n{'MODEL SIZE ANALYSIS':^{W}}\n{'═'*W}")
    print(f"{'Layer / Variable':<40} {'DType':<10} {'Params':>10} {'KB':>8}")
    print(f"{'─'*W}")

    total_bytes = 0
    for var in model.weights:
        dtype  = str(var.dtype).replace('tf.', '')
        params = tf.size(var).numpy()
        nbytes = params * dtype_bytes.get(dtype, 4)
        total_bytes += nbytes
        print(f"{var.name:<40} {dtype:<10} {params:>10,} {nbytes/1024:>8.2f}")

    print(f"{'═'*W}")
    print(f"{'Total Parameters':<40} {'':10} {model.count_params():>10,} {total_bytes/1024:>8.2f} KB")
    print(f"{'═'*W}\n")

In [ ]:
def plot_tr_val_accuracy_loss(history, save_path=None, dpi=300):

    epochs = np.arange(1, len(history.history['accuracy']) + 1)
    
    plt.rcParams.update({"font.family": "serif", "font.size": 11,
                         "axes.spines.top": False, "axes.spines.right": False})

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), facecolor="white")
    fig.suptitle("Training & Validation Metrics", fontsize=13, fontweight="bold")

    for ax, (train_key, val_key), title, ylabel, best_fn, best_label in zip(
        [ax1, ax2],
        [("accuracy", "val_accuracy"), ("loss", "val_loss")],
        ["(a)  Model Accuracy", "(b)  Model Loss"],
        ["Accuracy", "Loss"],
        [np.argmax, np.argmin],
        ["Best Val.", "Min Val."]
    ):
        train, val = history.history[train_key], history.history[val_key]
        best_epoch = int(best_fn(val)) + 1
        best_value = val[best_epoch - 1]

        ax.plot(epochs, train, color="#2563EB", linewidth=2, label="Training")
        ax.plot(epochs, val,   color="#DC2626", linewidth=2, linestyle="--", label="Validation")
        ax.axvline(best_epoch, color="#DC2626", linewidth=0.8, linestyle=":", alpha=0.7)
        ax.annotate(f"{best_label} {best_value:.4f}",
                    xy=(best_epoch, best_value),
                    xytext=(best_epoch + len(epochs) * 0.05, best_value),
                    fontsize=9, color="#DC2626",
                    arrowprops=dict(arrowstyle="->", color="#DC2626", lw=0.8),
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#DC2626", lw=0.8))
        ax.grid(color="#E5E7EB", linewidth=0.8)
        ax.set(title=title, xlabel="Epoch", ylabel=ylabel, xlim=(1, len(epochs)))
        ax.legend()

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.show()

### MAIN Usage

In [ ]:
# Loading the MIT-BIH Arrhythmia database

# MIT-BIH Arrhythmia database path
DATA_DIR = './MIT_BIH_Arrhythmia_Dataset'

print("Loading MIT-BIH Arrhythmia dataset ...")
X_ecg, X_rr, y = load_mitbih(DATA_DIR)
 
unique, counts = np.unique(y, return_counts=True)
print(f"  Total beats     : {len(y)}")
print(f"  Class counts    : {dict(zip(unique.tolist(), counts.tolist()))}")
print(f"  ECG array shape : {X_ecg.shape}")   # (N, 64, 1)
print(f"  RR  array shape : {X_rr.shape}")    # (N, 2)

In [ ]:
# Split the data
(Xe_tr, Xr_tr, y_tr,
 Xe_vl, Xr_vl, y_vl,
 Xe_te, Xr_te, y_te) = split_data(X_ecg, X_rr, y)
 
print(f"  Train / Val / Test : {len(y_tr)} / {len(y_vl)} / {len(y_te)}")

In [ ]:
# Class weights
cw = get_class_weights(y_tr)
print("  Class weights   :", {k: f"{v:.3f}" for k, v in cw.items()})

In [ ]:
# Save data for future use

# Define the save path
data_save_path = "processed_mit_bih_arrhythmia_dataset.npz"

# Save all split arrays into one compressed file
np.savez_compressed(
    data_save_path, 
    Xe_tr=Xe_tr, Xr_tr=Xr_tr, y_tr=y_tr,
    Xe_vl=Xe_vl, Xr_vl=Xr_vl, y_vl=y_vl,
    Xe_te=Xe_te, Xr_te=Xr_te, y_te=y_te
)

print(f"Data successfully saved to {data_save_path}")
print(f"File size: {os.path.getsize(data_save_path) / (1024*1024):.2f} MB")

#### Conv1D_2D_T Model

In [ ]:
# Build the best model (Conv1D_2D_T with trainable FIR)
conv1d_2d_t_model = build_conv1d_2d_model(trainable_fir=True)
conv1d_2d_t_model.summary()

In [ ]:
# Train the Conv1D_2D_T  model
print("Training Conv1D_2D_T ...")
history = train_model(conv1d_2d_t_model,
                  Xe_tr, Xr_tr, y_tr,
                  Xe_vl, Xr_vl, y_vl,
                  class_weights=cw,
                  epochs=50,
                  batch_size=256)

In [ ]:
# Model evaluation
model_evaluation(model = conv1d_2d_t_model, model_name = 'Conv1D_2D_T')

In [ ]:
# Model size analysis
get_model_size_analysis(conv1d_2d_t_model)

In [ ]:
# Plot training/validation accuracy and loss
plot_tr_val_accuracy_loss(history)

In [ ]:
# Save the model
conv1d_2d_t_model.save('stft_cnn_ecg_classifier_models/conv1d_2d_t_model.keras')

#### Conv1D_2D_T_FIR Model

In [ ]:
# Build the model (Conv1D_2D_T_FIR with  fixed FIR)
conv1d_2d_t_fir_model = build_conv1d_2d_model(trainable_fir=False)
conv1d_2d_t_fir_model.summary()

In [ ]:
# Train the Conv1D_2D_T_FIR  model
print("Training Conv1D_2D_T_FIR ...")
history = train_model(conv1d_2d_t_fir_model,
                  Xe_tr, Xr_tr, y_tr,
                  Xe_vl, Xr_vl, y_vl,
                  class_weights=cw,
                  epochs=50,
                  batch_size=256)

In [ ]:
# Model evaluation
model_evaluation(model=conv1d_2d_t_fir_model, model_name='Conv1D_2D_T_FIR')

In [ ]:
# Model size analysis
# get_model_size_analysis(conv1d_2d_t_fir_model)

In [ ]:
# Save the model
conv1d_2d_t_fir_model.save('stft_cnn_ecg_classifier_models/conv1d_2d_t_fir_model.keras')

In [ ]:
# Plot training/validation accuracy and loss
plot_tr_val_accuracy_loss(history)

#### Conv1D_T Model

In [ ]:
# Build the model (Conv1D_T with trainable FIR)
conv1d_t_model = build_conv1d_model(trainable_fir=True)
conv1d_t_model.summary()

In [ ]:
# Train the Conv1D_T  model
print("Training Conv1D_T ...")
history = train_model(conv1d_t_model,
                  Xe_tr, Xr_tr, y_tr,
                  Xe_vl, Xr_vl, y_vl,
                  class_weights=cw,
                  epochs=75,
                  batch_size=256)

In [ ]:
# Model evaluation
model_evaluation(model = conv1d_t_model, model_name = 'Conv1D_T')

In [ ]:
# Model size analysis
get_model_size_analysis(conv1d_t_model)

In [ ]:
# Save the model
conv1d_t_model.save('stft_cnn_ecg_classifier_models/conv1d_t_model.keras')

In [ ]:
# Plot training/validation accuracy and loss
plot_tr_val_accuracy_loss(history)

#### Conv1D_T_FIR Model

In [ ]:
# Build the model (Conv1D_T_FIR with fixed FIR)
conv1d_t_fir_model = build_conv1d_model(trainable_fir=False)

In [ ]:
# Train the Conv1D_T_FIR  model
print("Training Conv1D_T_FIR ...")
history = train_model(conv1d_t_fir_model,
                  Xe_tr, Xr_tr, y_tr,
                  Xe_vl, Xr_vl, y_vl,
                  class_weights=cw,
                  epochs=75,
                  batch_size=256)

In [ ]:
# Model evaluation
model_evaluation(model = conv1d_t_fir_model, model_name = 'Conv1D_T_FIR')

In [ ]:
# Model size analysis
#get_model_size_analysis(conv1d_t_fir_model)

In [ ]:
# Save the model
conv1d_t_model.save('stft_cnn_ecg_classifier_models/conv1d_t_fir_model.keras')

In [ ]:
# Plot training/validation accuracy and loss
#plot_tr_val_accuracy_loss(history)